# RAG End to End

Notebook for **Chapter 7** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 The Pipeline and Ingestion

## 7.2 Chunking Strategies

In [ ]:
import re, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
DOCS={
"refund_policy.md":"""# Refund policy
Refunds are available within 30 days of purchase.
Refunds are processed within 7 business days after approval.

# Exceptions
Gift cards cannot be refunded. Digital downloads are refundable only if not opened.""",
"shipping.md":"""# Shipping
Standard shipping takes 5 to 7 days. Express shipping takes 2 days.

# International
International orders ship in 10 to 14 days. Customs fees are paid by the buyer.""",
"support.md":"""# Support hours
Phone support is open 9am to 6pm on weekdays.

# Escalation
If a ticket is not answered in 24 hours, reply to the ticket to escalate it to a manager."""}
STOP={"the","is","a","an","do","does","how","are","can","i","of","to","in","on","if","it","be",
      "or","and","not","what","long","by","for","after","within"}
USE_STOP=True
def stem(w):
    if w.endswith("ed") and len(w)>4: w=w[:-2]
    elif w.endswith("s") and not w.endswith("ss") and len(w)>3: w=w[:-1]
    return w
def tok(s):
    ws=[stem(w) for w in re.findall(r"[a-z0-9]+",s.lower())]
    return [w for w in ws if not (USE_STOP and w in STOP)]
def chunk_fixed(text,size=12,overlap=3):
    w=text.split(); out=[]; i=0
    while i<len(w):
        out.append(" ".join(w[i:i+size])); i+=size-overlap
    return out
def chunk_sections(text):
    parts=re.split(r"\n(?=# )",text.strip()); return [p.strip() for p in parts]
def build(chunker):
    chunks=[(f,i,c) for f,t in DOCS.items() for i,c in enumerate(chunker(t))]
    vec=TfidfVectorizer(tokenizer=tok,token_pattern=None,sublinear_tf=True)
    M=vec.fit_transform([c[2] for c in chunks]); return chunks,vec,M
def retrieve(index,q,k=3):
    chunks,vec,M=index; s=(M@vec.transform([q]).T).toarray().ravel()
    return [(chunks[i],float(s[i])) for i in np.argsort(-s)[:k]]
def assemble(q,hits):
    ctx="\n".join(f"[{n}] ({c[0]}) {c[2]}" for n,(c,_) in enumerate(hits,1))
    return ("Answer ONLY from the context. Cite sources like [1]. If the context lacks the answer, say 'I do not know'.\n"
            f"Context:\n{ctx}\n\nQuestion: {q}\nAnswer:")
def fake_llm(prompt):
    """Stand-in for a real model: picks the context sentence with most word overlap with the question."""
    q=prompt.split("Question:")[1].split("\nAnswer:")[0]
    ctx=prompt.split("Context:\n")[1].split("\n\nQuestion:")[0]
    best,bs=None,0
    for line in ctx.split("\n"):
        m=re.match(r"\[(\d+)\]",line)
        if m: cur=m.group(1)
        for sent in re.split(r"(?<=[.])\s+",line):
            ov=len(set(tok(q))&set(tok(sent)))
            if ov>bs: best,bs=(sent,cur),ov
    if bs<2: return "I do not know."
    return re.sub(r"^\[\d+\]\s*\([^)]*\)\s*","",best[0])+f" [{best[1]}]"

In [ ]:
from rag import *
print(chunk_fixed(DOCS["shipping.md"])[:3])
print(len(chunk_fixed(DOCS["shipping.md"])),"fixed chunks vs",len(chunk_sections(DOCS["shipping.md"])),"section chunks")

['# Shipping Standard shipping takes 5 to 7 days. Express shipping takes', 'Express shipping takes 2 days. # International International orders ship in 10', 'ship in 10 to 14 days. Customs fees are paid by the']
4 fixed chunks vs 2 section chunks


In [ ]:
for name,ch in [("fixed",chunk_fixed),("sections",chunk_sections)]:
    idx=build(ch); q="How long does a refund take to process?"
    hits=retrieve(idx,q,2)
    print("==",name,"->",[(h[0][0],h[0][1],round(h[1],2)) for h in hits])

== fixed -> [('refund_policy.md', 1, 0.46), ('refund_policy.md', 0, 0.31)]
== sections -> [('refund_policy.md', 0, 0.43), ('shipping.md', 0, 0.28)]


## 7.3 Retrieval, Prompt Assembly, Generation and Citations

In [ ]:
idx=build(chunk_sections)
q="How long does a refund take to process?"
p=assemble(q,retrieve(idx,q,2)); print(p); print("LLM:",fake_llm(p))
q="What is the CEO's name?"; p=assemble(q,retrieve(idx,q,2)); print("LLM:",fake_llm(p))

Answer ONLY from the context. Cite sources like [1]. If the context lacks the answer, say 'I do not know'.
Context:
[1] (refund_policy.md) # Refund policy
Refunds are available within 30 days of purchase.
Refunds are processed within 7 business days after approval.
[2] (shipping.md) # Shipping
Standard shipping takes 5 to 7 days. Express shipping takes 2 days.

Question: How long does a refund take to process?
Answer:
LLM: Refunds are processed within 7 business days after approval. [1]
LLM: I do not know.


## 7.4 Advanced RAG

## 7.5 Failure Modes, Permissions and Operating RAG

## 7.6 Advanced RAG With Numbers: Multi-Query, Long Context Costs and Access Control

In [ ]:
from collections import defaultdict
lists = [["c4", "c1", "c7"], ["c1", "c4", "c9"], ["c1", "c7", "c2"]]
score = defaultdict(float)
for lst in lists:
    for rank, cid in enumerate(lst, 1):
        score[cid] += 1 / (60 + rank)
for cid, s in sorted(score.items(), key=lambda kv: -kv[1]):
    print(cid, round(s, 6))

c1 0.048916
c4 0.032522
c7 0.032002
c9 0.015873
c2 0.015873


In [ ]:
parents = {"P1": "Returns section (about 900 tokens)", "P2": "Delivery section (about 800 tokens)"}
children = {"c1": "P1", "c2": "P1", "c3": "P2", "c4": "P2"}
top_children = ["c2", "c1", "c4"]                     # from vector search on small chunks
sent = []
for c in top_children:
    p = children[c]
    if p not in sent:
        sent.append(p)
print("children retrieved:", len(top_children), "-> parents sent:", sent)
tokens_children = 3 * 150
tokens_parents = 900 + 800
print("tokens if children only:", tokens_children, " with parents:", tokens_parents)

children retrieved: 3 -> parents sent: ['P1', 'P2']
tokens if children only: 450  with parents: 1700


In [ ]:
price = 3.00 / 1e6
q = 1000
print("stuff:        ", 200_000 * q * price)
print("rag:          ", 4_000 * q * price)
print("stuff cached: ", 200_000 * q * price * 0.10)

stuff:         600.0
rag:           12.0
stuff cached:  60.0


In [ ]:
chunks = [
    {"id": "a", "groups": {"hr", "all"}, "text": "Holiday policy: 25 days"},
    {"id": "b", "groups": {"hr"}, "text": "Salary bands for managers"},
    {"id": "c", "groups": {"all"}, "text": "Office opens at 8am"},
]
def search(query_words, user_groups, k=2):
    allowed = [c for c in chunks if c["groups"] & user_groups]    # filter first
    scored = sorted(allowed, key=lambda c: -len(query_words & set(c["text"].lower().split())))
    return [c["id"] for c in scored[:k]]
q = {"salary", "bands", "managers", "office", "holiday"}
print("employee sees:", search(q, {"all"}))
print("hr sees:      ", search(q, {"hr", "all"}))

employee sees: ['a', 'c']
hr sees:       ['b', 'a']


## Chapter recap

## Practice set

Problem 7.P.1 (easy) What are the two phases of a RAG system? **Answer:** Offline indexing (parse, chunk, embed, store) and online answering (retrieve, assemble, generate).

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch07.html